# 🛡️ `ettin-pii`: Hands-On Interactive Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KalyanKS-NLP/ettin-pii/blob/main/docs/tutorial.ipynb)
[![PyPI Version](https://img.shields.io/pypi/v/ettin-pii.svg?color=blue)](https://pypi.org/project/ettin-pii/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

Welcome to the official interactive tutorial for **`ettin-pii`**, a fast, production-ready Python library for PII entity detection and  anonymization powered by the **Ettin Nemotron PII** model family.

### In this notebook, you will learn how to:
1. Install and initialize `PIIDetector` with different model sizes (`small`, `tiny`, `nano`).
2. Extract sensitive entities with character offsets and confidence scores.
3. Export detected entities directly to a Pandas DataFrame.
4. Apply all **4 redaction strategies** (`replace`, `mask`, `fake`, `custom`).
5. Compute quantitative privacy metrics and text density.
6. Process batches of text inputs with dataset-wide auditing.

## 1. Installation
Install `ettin-pii` and `pandas` directly into your notebook environment:

In [1]:
#!pip install ettin-pii pandas

## 2. Initialize the `PIIDetector`
You can choose between three official model sizes:
- `"small"` (~68.5M parameters, default) - Highest accuracy (~96.27% F1).
- `"tiny"` (~32M parameters) - Balanced speed and accuracy for real-time CPU use.
- `"nano"` (~17.5M parameters) - Ultra-fast inference for serverless and edge workloads.

In [2]:
from ettin_pii import PIIDetector

# Initialize with the default 'small' model
# (You can also try model='tiny' or model='nano')
detector = PIIDetector(model="small", confidence_threshold=0.55)
print("PIIDetector initialized and ready!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/7.45k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  137MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/120 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

PIIDetector initialized and ready!


## 3. Single-Text Detection
Let's test the detector on a realistic customer support message containing multiple sensitive entity types:

In [3]:
sample_text = (
    "Customer Alex Murphy (SSN: 987-65-4321) requested a password reset. "
    "Contact them at alex.murphy@corp.org or call 415-555-0199."
)

result = detector.detect(sample_text)

print(f"Found {result.num_entities} PII entities:\n")
for ent in result.entities:
    print(f"  • [{ent.label}] '{ent.text}' (Score: {ent.score:.2%}, Offsets: {ent.start}-{ent.end})")

Found 5 PII entities:

  • [first_name] 'Alex' (Score: 98.65%, Offsets: 9-13)
  • [last_name] 'Murphy' (Score: 78.82%, Offsets: 14-20)
  • [ssn] '987-65-4321' (Score: 96.42%, Offsets: 27-38)
  • [email] 'alex.murphy@corp.org' (Score: 99.12%, Offsets: 84-104)
  • [phone_number] '415-555-0199' (Score: 99.86%, Offsets: 113-125)


## 4. Convert Entities to a Pandas DataFrame
You can convert the detection output into a clean Pandas DataFrame in one method call:

In [4]:
df = result.to_dataframe()
df

,text,label,score,start,end,length
0,Alex,first_name,0.9865,9,13,4
1,Murphy,last_name,0.7882,14,20,6
2,987-65-4321,ssn,0.9642,27,38,11
3,alex.murphy@corp.org,email,0.9912,84,104,20
4,415-555-0199,phone_number,0.9986,113,125,12


## 5. Exploring the 4 Anonymization Strategies
`ettin-pii` provides 4 flexible redaction strategies:

In [5]:
print("=== ORIGINAL TEXT ===")
print(sample_text)

print("\n=== 1. REPLACE (Tags) ===")
print(result.redact(strategy="replace"))

print("\n=== 2A. MASK (Full Asterisk) ===")
print(result.redact(strategy="mask", mask_mode="full"))

print("\n=== 2B. MASK (First Character Visible) ===")
print(result.redact(strategy="mask", mask_mode="first_only"))

print("\n=== 2C. MASK (Alternate Characters) ===")
print(result.redact(strategy="mask", mask_mode="alternate"))

print("\n=== 3. FAKE (Format-Preserving Synthetic Scramble) ===")
print(result.redact(strategy="fake"))

print("\n=== 4. CUSTOM (Dictionary Mapping with Automatic Fallback) ===")
custom_rules = {
    "email": "[CONFIDENTIAL_EMAIL]",
    "first_name": "[ANONYMOUS_USER]",
}
print(result.redact(strategy="custom", mapping=custom_rules))

=== ORIGINAL TEXT ===
Customer Alex Murphy (SSN: 987-65-4321) requested a password reset. Contact them at alex.murphy@corp.org or call 415-555-0199.

=== 1. REPLACE (Tags) ===
Customer <FIRST_NAME> <LAST_NAME> (SSN: <SSN>) requested a password reset. Contact them at <EMAIL> or call <PHONE_NUMBER>.

=== 2A. MASK (Full Asterisk) ===
Customer **** ****** (SSN: ***********) requested a password reset. Contact them at ******************** or call ************.

=== 2B. MASK (First Character Visible) ===
Customer A*** M***** (SSN: 9**********) requested a password reset. Contact them at a******************* or call 4***********.

=== 2C. MASK (Alternate Characters) ===
Customer A*e* M*r*h* (SSN: 9*7-6*-*3*1) requested a password reset. Contact them at a*e*.*u*p*y@c*r*.*r* or call 4*5-5*5-0*9*.

=== 3. FAKE (Format-Preserving Synthetic Scramble) ===
Customer Hdky Ckkxcs (SSN: 202-84-9827) requested a password reset. Contact them at uetr.vnakmg@vudk.myt or call 355-642-8295.

=== 4. CUSTOM (Di

## 6. Privacy Auditing & Quantitative Metrics
Use `.statistics` to calculate the percentage of sensitive text content and average confidence:

In [6]:
import json

stats = result.statistics
print(json.dumps(stats, indent=2))
print(f"\nPrivacy Exposure Density: {stats['density']:.2%} of text characters contain PII.")

{
  "total_entities": 5,
  "unique_entity_types": 5,
  "entity_counts": {
    "first_name": 1,
    "last_name": 1,
    "ssn": 1,
    "email": 1,
    "phone_number": 1
  },
  "density": 0.4206,
  "average_confidence": 0.9457
}

Privacy Exposure Density: 42.06% of text characters contain PII.


## 7. High-Volume Batch Processing
When you pass a list of strings, `.detect()` automatically returns a `BatchDetectionResult` with dataset-wide auditing and list-like slicing:

In [7]:
support_tickets = [
    "Hi, I am Sarah Connor. Reach me at sarah.c@sky.net or call 310-555-0144.",
    "Issue with checkout button on page 3. HTTP 500 error logged.",
    "Please send verification pin to dave.miller@fintech.io.",
    "Server 192.168.1.100 assigned to Dave Murphy (SSN: 987-65-4321).",
    "Nightly database backup completed with zero errors.",
]

batch_results = detector.detect(support_tickets, batch_size=2)

print(f"Processed {len(batch_results)} tickets.\n")
print("=== BATCH AUDIT METRICS ===")
print(json.dumps(batch_results.statistics, indent=2))

Processed 5 tickets.

=== BATCH AUDIT METRICS ===
{
  "total_texts": 5,
  "texts_with_pii": 3,
  "total_entities": 9,
  "unique_entity_types": 6,
  "entity_counts": {
    "first_name": 2,
    "last_name": 2,
    "email": 2,
    "phone_number": 1,
    "ipv4": 1,
    "ssn": 1
  }
}


## 8. Combined Batch DataFrame
Export all detected entities across every document in the batch into a single traceable table:

In [8]:
batch_df = batch_results.to_dataframe()
batch_df[["text_index", "label", "text", "score", "source_text"]]

,text_index,label,text,score,source_text
0,0,first_name,Sarah,0.9909,"Hi, I am Sarah Connor. Reach me at sarah.c@sky..."
1,0,last_name,Connor,0.6618,"Hi, I am Sarah Connor. Reach me at sarah.c@sky..."
2,0,email,sarah.c@sky.net,0.9961,"Hi, I am Sarah Connor. Reach me at sarah.c@sky..."
3,0,phone_number,310-555-0144,0.9993,"Hi, I am Sarah Connor. Reach me at sarah.c@sky..."
4,2,email,dave.miller@fintech.io,0.9840,Please send verification pin to dave.miller@fi...
5,3,ipv4,192.168.1.100,0.9398,Server 192.168.1.100 assigned to Dave Murphy (...
6,3,first_name,Dave,0.9374,Server 192.168.1.100 assigned to Dave Murphy (...
7,3,last_name,Murphy,0.7780,Server 192.168.1.100 assigned to Dave Murphy (...
8,3,ssn,987-65-4321,0.9639,Server 192.168.1.100 assigned to Dave Murphy (...


## 9. Batch Redaction
Anonymize all tickets at once with your preferred redaction strategy:

In [9]:
clean_tickets = batch_results.redact(strategy="replace")
for i, clean in enumerate(clean_tickets):
    print(f"Ticket #{i+1}: {clean}")

Ticket #1: Hi, I am <FIRST_NAME> <LAST_NAME>. Reach me at <EMAIL> or call <PHONE_NUMBER>.
Ticket #2: Issue with checkout button on page 3. HTTP 500 error logged.
Ticket #3: Please send verification pin to <EMAIL>.
Ticket #4: Server <IPV4> assigned to <FIRST_NAME> <LAST_NAME> (SSN: <SSN>).
Ticket #5: Nightly database backup completed with zero errors.


## 10. Summary & Useful Links

You now have all the tools to detect, audit, and anonymize sensitive data at scale!

- **PyPI Package:** [https://pypi.org/project/ettin-pii/](https://pypi.org/project/ettin-pii/)
- **GitHub Repository:** [https://github.com/KalyanKS-NLP/ettin-pii](https://github.com/KalyanKS-NLP/ettin-pii)
- **Hugging Face Models:** [https://huggingface.co/kalyan-ks](https://huggingface.co/kalyan-ks)
- **Full User Guide:** [`docs/user_guide.md`](https://github.com/KalyanKS-NLP/ettin-pii/blob/main/docs/user_guide.md)